<div style="font-family: SimSun, 宋体, serif; font-size: 10.5pt; line-height: 1.5; margin: 0; text-align: justify;">Task 2：加载已保存股价数据，检查缺失值，计算描述性统计量，计算 RSI、MACD、布林带和 OBV，并绘图。</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimSun']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../task1_daily_data.csv')
df = df.sort_values('trade_date')
print('缺失值：')
print(df.isna().sum())
print('描述性统计：')
display(df.describe())

close = df['close']
delta = close.diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df['RSI'] = 100 - 100 / (1 + gain / loss)

ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
df['DIF'] = ema12 - ema26
df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
df['MACD'] = 2 * (df['DIF'] - df['DEA'])

df['BOLL_MID'] = close.rolling(20).mean()
df['BOLL_STD'] = close.rolling(20).std()
df['BOLL_UPPER'] = df['BOLL_MID'] + 2 * df['BOLL_STD']
df['BOLL_LOWER'] = df['BOLL_MID'] - 2 * df['BOLL_STD']

direction = close.diff().apply(lambda x: 1 if x > 0 else (-1 if x < 0 else 0))
df['OBV'] = (direction * df['vol']).fillna(0).cumsum()

df[['close', 'BOLL_UPPER', 'BOLL_MID', 'BOLL_LOWER']].plot(figsize=(12, 5), title='布林带')
plt.show()
df[['RSI']].plot(figsize=(12, 3), title='RSI')
plt.show()
df[['DIF', 'DEA', 'MACD']].plot(figsize=(12, 4), title='MACD')
plt.show()
df[['OBV']].plot(figsize=(12, 3), title='OBV 能量潮')
plt.show()